In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
import copy

In [2]:
num_classes = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batchsize = 16

In [3]:
class EEGDataset(Dataset):
    def __init__(self, data_path, labels_path):
        self.data = np.load(data_path)
        self.labels = np.load(labels_path)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get the EEG data and corresponding label
        eeg = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return eeg, label

# Example usage
train_dataset = EEGDataset("eeg_dataset/train_epochs.npy", "eeg_dataset/train_labels.npy")
val_dataset = EEGDataset("eeg_dataset/val_epochs.npy", "eeg_dataset/val_labels.npy")
test_dataset = EEGDataset("eeg_dataset/test_epochs.npy", "eeg_dataset/test_labels.npy")

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batchsize, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

print(len(train_loader))
# Check one batch
for eeg_batch, label_batch in train_loader:
    print("EEG Batch Shape:", eeg_batch.shape)  # (batch_size, 14, 640)
    n_channels = eeg_batch.shape[1]
    samples_per_data = eeg_batch.shape[2]
    print("Label Batch Shape:", label_batch.shape)  # (batch_size,)
    break


79
EEG Batch Shape: torch.Size([16, 8, 1088])
Label Batch Shape: torch.Size([16])


In [4]:
sampling_rate = 256
print(n_channels)
print(samples_per_data)

8
1088


In [5]:
# classifier
class EEGNet(nn.Module):
    def __init__(self, n_class, n_channels, total_samples, sampling_rate, F1 = 8, F2 = 16, D = 2):
        super(EEGNet, self).__init__()
        
        self.Encoder = nn.Sequential(
            
            # Block 1
            nn.Conv2d(in_channels=1, out_channels=F1, kernel_size=(1, sampling_rate//2), 
                      padding="same", bias=False), # (8, 8, 1076)
            nn.BatchNorm2d(num_features=F1),
            nn.Conv2d(in_channels=F1, groups=F1, out_channels=D*F1, kernel_size=(n_channels, 1), bias=False), # (2*8, 1, 1076)
            nn.BatchNorm2d(num_features=D*F1),
            nn.ELU(),
            nn.AvgPool2d(kernel_size = (1, 4)), # (2*8, 1, 269)
            nn.Dropout(0.25),

            # Block 2
            nn.Conv2d(in_channels=D*F1, groups=D*F1, out_channels=D*F1, kernel_size=(1, sampling_rate//8), 
                      padding="same", bias=False), # (2*8, 1, 269)
            nn.Conv2d(in_channels=D*F1, out_channels=F2, kernel_size=(1, 1), groups=1, bias = False), # (16, 1, 269)
            nn.BatchNorm2d(num_features=F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size = (1, 8)), # (16, 1, 33)
            nn.Dropout(0.25),
            # Flatten
            nn.Flatten()
        )
        
        self.Decoder = nn.Sequential(
            
            ### Reverse Block 2
            nn.Dropout(0.25),
            nn.Unflatten(1, (F2, 1, total_samples//32)),  # 16, 1, 33
            nn.Upsample(scale_factor=(1, 8)), # 16, 1, 264
            nn.ELU(),
            nn.BatchNorm2d(num_features=F2),
            nn.Conv2d(in_channels=F1*D, out_channels=F2, kernel_size=(1, 1), groups = 1, bias = False),
            nn.Conv2d(in_channels=F1*D, out_channels=F1*D, kernel_size=(1, sampling_rate//8), groups = F1*D, padding = 'same', bias = False),
            
            ## Reverse Block 1
            nn.Dropout(0.25),
            nn.Upsample(scale_factor=(1, 4)),
            nn.ELU(),
            nn.BatchNorm2d(num_features=F1 * D),
            nn.ConvTranspose2d(in_channels=D*F1, out_channels=F1, kernel_size=(n_channels, 1), groups = F1, bias = False), # Spatial
            nn.BatchNorm2d(num_features=F1),
            nn.Conv2d(in_channels=F1, out_channels=1, kernel_size=(1, sampling_rate//2), padding="same", bias=False),
        )

    def forward(self, x):
        encoded = self.Encoder(x)
        decoded = self.Decoder(encoded)
        return encoded, decoded

In [54]:
model = EEGNet(12, n_channels, samples_per_data, sampling_rate).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)   # optimize all parameters
criterion = nn.MSELoss()

best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 1000

for epoch in range(100):
    
    # Train the data
    model.train()
    running_loss = 0.0
    for inputs, label in train_loader:
        # Inputs are in shape (batch_size, channels, timepoints)
        # Need to transform to (batch_size, 1, channels, timepoints)
        inputs = inputs.unsqueeze(1).to(device)
        label = label.to(device)
        
        encoded_info, decoded_info = model(inputs)
        loss = criterion(decoded_info, inputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # Validate Data
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, label in val_loader:
            # Inputs are in shape (batch_size, channels, timepoints)
            # Need to transform to (batch_size, 1, channels, timepoints)
            inputs = inputs.unsqueeze(1).to(device)
            label = label.to(device)
            
            encoded_info, decoded_info = model(inputs)
            loss = criterion(decoded_info, inputs)
            val_loss += loss.item()

    # Print to see status
    print(f'Epoch {epoch+1}, Training Loss: {running_loss/len(train_loader)}, Validation Acc: {val_loss/len(val_loader)}%')

    if val_loss < best_acc:
        best_acc = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())

Epoch 1, Training Loss: 0.9896780124193505, Validation Acc: 0.7997970160316018%
Epoch 2, Training Loss: 0.7707324593882018, Validation Acc: 0.6424753140000736%
Epoch 3, Training Loss: 0.6957275942911075, Validation Acc: 0.6009965819471023%


KeyboardInterrupt: 

In [53]:
model.load_state_dict(best_model_wts)
# test
with torch.no_grad():
    for eeg, labels in test_loader:
        eeg = eeg.unsqueeze(1).to(device)
        labels = labels.to(device)
        encoded, decoded = model(eeg)
        
        example_eeg = eeg.detach().cpu().numpy()[13][0]
        decoded_eeg = decoded.detach().cpu().numpy()[13][0]
        
        plt.plot(example_eeg[5])
        plt.plot(decoded_eeg[5])
        
        break

RuntimeError: Given groups=1, weight of size [16, 32, 1, 1], expected input[16, 16, 1, 272] to have 32 channels, but got 16 channels instead

In [45]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

test_latent = []
test_labels = []

with torch.no_grad():
    for eeg, labels in test_loader:
        eeg = eeg.unsqueeze(1).to(device)
        encoded = model.Encoder(eeg)
        
       
        temp_latent =  encoded.detach().cpu().numpy()
        temp_labels = labels.detach().cpu().numpy()
        
        
        for i in range(len(temp_latent)):
            test_latent.append(temp_latent[i])
            test_labels.append(temp_labels[i])
        
        example_eeg = eeg.detach().cpu().numpy()[10][0]
        decoded_eeg = decoded.detach().cpu().numpy()[10][0]
        

In [51]:
test_latent = np.array(test_latent)
kmeans = KMeans(n_clusters=12, max_iter=1000, n_init=20)
clusters = kmeans.fit(test_latent)

for i in range(len(test_latent)):
    cluster_belong = kmeans.predict(test_latent[i].reshape(1, -1))
    print(f'Cluster {cluster_belong}, label={test_labels[i]}')

Cluster [0], label=4
Cluster [6], label=5
Cluster [5], label=10
Cluster [7], label=6
Cluster [0], label=11
Cluster [2], label=10
Cluster [1], label=3
Cluster [5], label=4
Cluster [3], label=4
Cluster [6], label=10
Cluster [8], label=7
Cluster [11], label=1
Cluster [3], label=6
Cluster [6], label=1
Cluster [6], label=11
Cluster [6], label=8
Cluster [9], label=0
Cluster [0], label=11
Cluster [1], label=2
Cluster [0], label=11
Cluster [5], label=10
Cluster [9], label=7
Cluster [6], label=8
Cluster [11], label=11
Cluster [1], label=2
Cluster [8], label=7
Cluster [11], label=1
Cluster [3], label=0
Cluster [4], label=8
Cluster [6], label=4
Cluster [0], label=8
Cluster [9], label=0
Cluster [3], label=10
Cluster [5], label=2
Cluster [0], label=11
Cluster [0], label=8
Cluster [10], label=10
Cluster [2], label=3
Cluster [6], label=8
Cluster [4], label=9
Cluster [4], label=9
Cluster [2], label=7
Cluster [6], label=11
Cluster [6], label=8
Cluster [0], label=11
Cluster [9], label=7
Cluster [10], la